In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

load_dotenv()
data_pth = Path(os.getenv("DATA_PATH", "data"))
train_csv_pth = f"{data_pth}/train.csv"
train_img_pth = Path(f"{data_pth}/train_images")

train_csv = pd.read_csv(train_csv_pth)
train_csv = train_csv.copy()

In [2]:
imageide = train_csv["ImageId"].value_counts()
duplicates_combined = imageide.index.tolist()
train_file_name = [
    file_path.name for file_path in train_img_pth.iterdir() if file_path.is_file()
]

# Grouped defect profiles
class_ = []
for item in duplicates_combined:
    duplicate_statuses = train_csv[["ImageId", "ClassId"]].loc[
        train_csv["ImageId"] == item
    ]
    class_.append((item, "&".join(map(str, duplicate_statuses.ClassId.tolist()))))

# Matched train.csv with train_images and assigned "Clean" profile to train_images without defects
for obj in train_file_name:
    if obj not in duplicates_combined:
        class_.append((obj, "Clean"))

In [3]:
len(train_file_name)

12568

In [4]:
import csv

headers = ["ImageId", "Class"]
file_pth = Path(os.getenv("CURRENT_FILE_PATH", ""))
with open("train_folds.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(headers)
    writer.writerows(class_)

In [5]:
train_fold = pd.read_csv("train_folds.csv")
train_fold["Fold"] = -1

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(train_fold, train_fold["Class"])):
    train_fold.loc[val_idx, "Fold"] = fold

train_fold.to_csv("train_folds.csv", index=False)

c:\Users\Chinedu\Documents\segmentation_project\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [ ]:
def split_df_on_fold(
    val_fold_no: int,
    train_fold_df: pd.DataFrame,
    train_csv_df: pd.DataFrame,
):
    train_fold = train_fold_df.copy()
    train_csv = train_csv_df.copy()

    clean = train_fold[["ImageId", "ClassId"]].loc[train_fold["ClassId"] == "Clean"]
    concat_train_csv = pd.concat([train_csv, clean], ignore_index=True)
    concat_train_csv = concat_train_csv.merge(train_fold, how="inner", on="ImageId")
    train_df = concat_train_csv.loc[concat_train_csv["Fold"] == val_fold_no]
    val_df = concat_train_csv.loc[concat_train_csv["Fold"] != val_fold_no]
    return train_df, val_df

In [7]:
train_fold.rename(columns={"Class": "ClassId"}, inplace=True)

In [ ]:
clean = train_fold[["ImageId", "ClassId"]].loc[train_fold["ClassId"] == "Clean"]
concat_train_csv = pd.concat([train_csv, clean], ignore_index=True)
concat_train_csv.tail()

,ImageId,ClassId,EncodedPixels
12992,ffa8210a1.jpg,Clean,NaN
12993,ffaa05016.jpg,Clean,NaN
12994,ffd0223a7.jpg,Clean,NaN
12995,ffe93442c.jpg,Clean,NaN
12996,fff0295e1.jpg,Clean,NaN


In [10]:
concat_train_csv = concat_train_csv.merge(train_fold, how="inner", on="ImageId")
concat_train_csv.head()

,ImageId,ClassId_x,EncodedPixels,ClassId_y,Fold
0,0002cc93b.jpg,1,29102 12 29346 24 29602 24 29858 24 30114 24 3...,1,2
1,0007a71bf.jpg,3,18661 28 18863 82 19091 110 19347 110 19603 11...,3,0
2,000a4bcdd.jpg,1,37607 3 37858 8 38108 14 38359 20 38610 25 388...,1,1
3,000f6bf48.jpg,4,131973 1 132228 4 132483 6 132738 8 132993 11 ...,4,0
4,0014fce06.jpg,3,229501 11 229741 33 229981 55 230221 77 230468...,3,4


In [13]:
train_df = concat_train_csv.loc[concat_train_csv["Fold"] == 1]
train_df

,ImageId,ClassId_x,EncodedPixels,ClassId_y,Fold
2,000a4bcdd.jpg,1,37607 3 37858 8 38108 14 38359 20 38610 25 388...,1,1
18,008ef3d74.jpg,1,356336 4 356587 11 356838 18 357089 25 357340 ...,1&2,1
19,008ef3d74.jpg,2,375439 5 375687 14 375935 24 376182 34 376430 ...,1&2,1
20,0095cd374.jpg,3,10674 18 10926 48 11178 52 11430 56 11682 60 1...,3,1
24,00bc01bfe.jpg,1,212941 6 213193 18 213446 25 213701 26 213956 ...,1,1
...,...,...,...,...,...
12959,fdfbe4d02.jpg,Clean,NaN,Clean,1
12973,feb7f851f.jpg,Clean,NaN,Clean,1
12989,ff8c44174.jpg,Clean,NaN,Clean,1
12992,ffa8210a1.jpg,Clean,NaN,Clean,1


In [14]:
val_df = concat_train_csv.loc[concat_train_csv["Fold"] != 1]
val_df

,ImageId,ClassId_x,EncodedPixels,ClassId_y,Fold
0,0002cc93b.jpg,1,29102 12 29346 24 29602 24 29858 24 30114 24 3...,1,2
1,0007a71bf.jpg,3,18661 28 18863 82 19091 110 19347 110 19603 11...,3,0
3,000f6bf48.jpg,4,131973 1 132228 4 132483 6 132738 8 132993 11 ...,4,0
4,0014fce06.jpg,3,229501 11 229741 33 229981 55 230221 77 230468...,3,4
5,0025bde0c.jpg,3,8458 14 8707 35 8963 48 9219 71 9475 88 9731 8...,3&4,0
...,...,...,...,...,...
12990,ff96da52e.jpg,Clean,NaN,Clean,0
12991,ffa55c731.jpg,Clean,NaN,Clean,0
12993,ffaa05016.jpg,Clean,NaN,Clean,2
12994,ffd0223a7.jpg,Clean,NaN,Clean,3
